In [9]:
import pandas as pd 
from torchmetrics.text.rouge import ROUGEScore
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

In [7]:
data_path = "logger/gemma-2-2b+domain_masked_wrong_answer/result.csv"
model_name = "google/gemma-2-2b-it"

In [3]:
df = pd.read_csv(data_path)

In [5]:
df.head()

,Unnamed: 0,Question,Answer,Label,rouge_score
0,0,<bos><start_of_turn>user\nHãy điền vào trong d...,.\nmodel\nA: Báo với cô giáo chủ nhiệm để tìm ...,Chạy đi chỗ khác chơi.,0.357143
1,1,<bos><start_of_turn>user\nHãy điền vào trong d...,.\nmodel\nA: **Đưa em bé đó đến bệnh viện và g...,Lờ đi chỗ khác và coi như không biết.,0.121212
2,2,<bos><start_of_turn>user\nHãy điền vào trong d...,.\nmodel\nA: **Gió chiều nào theo chiều ấy nói...,Không tôn trọng lẽ phải.,0.206897
3,3,<bos><start_of_turn>user\nHãy điền vào trong d...,".\nmodel\nC: **Làm giàu bằng chính sức lực, tà...",Nhặt được của rơi trả lại cho người bị mất,0.315789
4,4,<bos><start_of_turn>user\nHãy điền vào trong d...,.\nmodel\nB: Liêm khiết. \n \n,Công bằng.,0.200000


In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [10]:
def normalizer(text):
    text = re.sub(r"[^a-z0-9àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]+", " ", text.lower())
    text = text.strip(" ")
    return text

rouge = ROUGEScore(rouge_keys="rougeL", tokenizer=tokenizer.tokenize, use_stemmer= False, normalizer=normalizer)

In [11]:
res = []
for _, row in df.iterrows():
    target = row["Label"]
    prediction = row["Answer"]
    score = rouge(prediction, target)
    res.append(score)

In [12]:
df["check_score"] = res

In [17]:
df["check_score"] = df["check_score"].apply(lambda x: x["rougeL_fmeasure"].item())

In [18]:
df["check_score"].mean()

0.20326764406048478

In [19]:
df["rouge_score"].mean()

0.30902643013565406